# Extraction du Dataset
Pipeline : Dataset PyTorch → fine-tuning de modèles pré-entraînés → évaluation

## 1. Imports

In [1]:
from pathlib import Path
import random
import torchvision.models as models
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

random.seed(42)
torch.manual_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")

Device : cuda


## 2. Dataframe labels

In [2]:
DATA_DIR = Path("data")
assert DATA_DIR.exists(), f"DATA_DIR introuvable — CWD : {Path.cwd()}"

records = []
for img_path in DATA_DIR.rglob("*.jpg"):
    relative_parts = img_path.relative_to(DATA_DIR).parts
    if "cancer" in relative_parts:
        label = "cancer"
    elif "normal" in relative_parts:
        label = "normal"
    else:
        label = "unlabeled"
    records.append({"path": img_path, "label": label})

df = pd.DataFrame(records)
print(f"Total images : {len(df)}")
print(df["label"].value_counts())

Total images : 1506
label
unlabeled    1406
cancer         50
normal         50
Name: count, dtype: int64


## 3. Dataset PyTorch

In [3]:
LABEL_MAP = {"cancer": 1, "normal": 0, "unlabeled": -1}

# Inference transforms — no augmentation, just resize + ImageNet normalization
TRANSFORMS = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

class full_MRI(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(row["path"]) as img:
            if self.transform:
                img = self.transform(img)
        label = LABEL_MAP[row["label"]]
        return img, label

dataset = full_MRI(df, transform=TRANSFORMS)
loader  = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Total images : {len(dataset)}")

Total images : 1506


# Extraction via RESNET50

In [4]:
# Load ResNet50 pretrained, remove the final classification layer
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
resnet.fc = torch.nn.Identity()  # output: 2048-dim embedding
resnet = resnet.to(DEVICE)
resnet.eval()

all_embeddings = []
all_labels = []

with torch.no_grad():
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        embeddings = resnet(imgs)           # (batch, 2048)
        all_embeddings.append(embeddings.cpu().numpy())
        all_labels.append(labels.numpy())

embeddings_array = np.concatenate(all_embeddings, axis=0)  # (1506, 2048)
labels_array     = np.concatenate(all_labels,     axis=0)  # (1506,)

print(f"Embeddings shape : {embeddings_array.shape}")
print(f"Labels shape     : {labels_array.shape}")

Embeddings shape : (1506, 2048)
Labels shape     : (1506,)


Creation d'un dataframe avec les embeddings et les labels:

In [5]:
resnetdf = pd.DataFrame(embeddings_array)
resnetdf['label'] = labels_array
resnetdf['image_path'] = df['path']


In [6]:
resnetdf

,0,1,2,3,4,5,6,7,8,9,...,2040,2041,2042,2043,2044,2045,2046,2047,label,image_path
0,0.000000,0.033223,0.030290,0.006026,0.000000,0.001288,0.399393,0.000000,0.404415,0.000000,...,0.116863,0.082014,0.011174,0.012793,1.722093,0.000000,0.007299,0.104921,-1,data\sans_label\001b158a-7af8-451e-bf31-3a9116...
1,0.007491,0.001147,0.022209,0.000000,0.019319,0.000000,0.202351,0.059050,0.000000,0.000000,...,0.261960,0.000000,0.131598,0.058813,0.560453,0.019349,0.078556,0.000000,-1,data\sans_label\00366e8d-5520-4d3c-a70b-91a7ee...
2,0.023812,0.006307,0.166584,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.049505,0.010434,0.004079,0.003989,0.649365,0.000000,0.037040,0.000000,-1,data\sans_label\00455a62-f79f-4072-9a23-4951e7...
3,0.028151,0.000000,0.066577,0.000000,0.026256,0.000000,0.534010,0.016958,0.087047,0.000000,...,0.098995,0.002062,0.034830,0.021874,0.675863,0.025730,0.000000,0.161040,-1,data\sans_label\004ce5f5-ca6b-490f-9b2f-c322c1...
4,0.000000,0.205159,0.394879,0.000000,0.000000,0.024679,0.380019,0.000000,0.785505,0.000000,...,0.242116,0.104951,0.055924,0.166249,0.192706,0.000000,0.000000,0.050970,-1,data\sans_label\005d9a37-8894-4eb5-8367-1015d4...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1501,0.050378,0.000000,0.086331,0.000000,0.000000,0.000000,0.033195,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.008170,0.000000,0.559177,0.000000,0.000000,0.034471,0,data\avec_labels\normal\cfda6929-14c5-41b9-b75...
1502,0.000000,0.000000,0.106654,0.000069,0.428068,0.008417,0.069095,0.036308,0.000000,0.000262,...,0.000000,0.000000,0.027075,0.000000,1.559866,0.000000,0.000000,0.028039,0,data\avec_labels\normal\d86f39ec-55a1-4484-bcf...
1503,0.000000,0.031213,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.001149,0.000000,...,0.000000,0.000000,0.007162,0.050631,0.408271,0.000000,0.023442,0.042651,0,data\avec_labels\normal\defdbef3-bea2-4f32-9d9...
1504,0.060650,0.000000,0.027905,0.000000,0.000000,0.005681,0.002483,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.142223,0.000000,0.421681,0.000000,0.014103,0.039110,0,data\avec_labels\normal\e30c2ce4-50bd-44a9-8db...


# Extraction via EfficientNet_b4

In [7]:

# Load efficientnet pretrained, remove the final classification layer
efficientnet = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
efficientnet.classifier = torch.nn.Identity()  # "classifier" et non "fc"
efficientnet = efficientnet.to(DEVICE)
efficientnet.eval()

all_embeddings = []
all_labels = []

with torch.no_grad():
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        embeddings = efficientnet(imgs)           # (batch, 2048)
        all_embeddings.append(embeddings.cpu().numpy())
        all_labels.append(labels.numpy())

embeddings_array = np.concatenate(all_embeddings, axis=0)  # (1506, 2048)
labels_array     = np.concatenate(all_labels,     axis=0)  # (1506,)

print(f"Embeddings shape : {embeddings_array.shape}")
print(f"Labels shape     : {labels_array.shape}")

Embeddings shape : (1506, 1792)
Labels shape     : (1506,)


In [8]:
efnetdf = pd.DataFrame(embeddings_array)
efnetdf['label'] = labels_array
efnetdf['image_path'] = df['path']

In [9]:
efnetdf

,0,1,2,3,4,5,6,7,8,9,...,1784,1785,1786,1787,1788,1789,1790,1791,label,image_path
0,-0.025993,-0.042504,0.064541,-0.041966,0.052763,0.233549,0.094920,0.011328,0.003923,0.052546,...,0.020870,0.146621,-0.017939,-0.059028,0.282665,0.080997,0.140180,-0.134590,-1,data\sans_label\001b158a-7af8-451e-bf31-3a9116...
1,-0.037984,-0.048494,0.044174,0.020470,0.110816,0.287991,-0.008233,-0.066672,-0.065927,-0.052903,...,0.118220,0.087558,0.066695,-0.050155,0.277727,0.290887,-0.005268,-0.068579,-1,data\sans_label\00366e8d-5520-4d3c-a70b-91a7ee...
2,0.042334,0.110480,0.001232,0.019372,0.121153,0.278846,0.032056,0.205399,0.027532,-0.016283,...,0.157080,0.178464,0.026619,-0.035401,0.192710,0.180350,0.051158,-0.074635,-1,data\sans_label\00455a62-f79f-4072-9a23-4951e7...
3,-0.121805,0.023672,0.296081,0.024863,0.036877,0.196573,0.029901,0.038729,-0.041944,-0.107116,...,0.097943,0.297791,-0.035078,-0.036565,0.119878,0.087571,-0.064634,-0.118060,-1,data\sans_label\004ce5f5-ca6b-490f-9b2f-c322c1...
4,-0.003182,-0.120139,-0.039899,-0.033802,0.042015,0.206961,0.119297,0.063566,-0.085056,0.126738,...,0.109947,0.115744,-0.018765,-0.099622,0.286492,0.161195,0.397144,-0.072973,-1,data\sans_label\005d9a37-8894-4eb5-8367-1015d4...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1501,0.008825,0.045315,0.024698,0.005651,0.081029,0.237376,0.004415,0.003247,-0.053804,0.003394,...,0.137376,0.143957,0.004815,-0.108104,-0.010168,0.069391,0.068793,-0.061641,0,data\avec_labels\normal\cfda6929-14c5-41b9-b75...
1502,-0.085211,-0.040787,0.002099,0.005782,0.134183,0.082996,0.052482,0.033138,0.073488,-0.080448,...,0.149702,0.191078,-0.056966,-0.188925,-0.084784,-0.017290,-0.036144,-0.032449,0,data\avec_labels\normal\d86f39ec-55a1-4484-bcf...
1503,-0.045674,-0.008590,0.121327,0.006675,0.208660,0.202928,0.028440,0.083404,0.114125,-0.080588,...,0.027897,0.086875,0.021427,-0.164833,0.132899,0.116851,0.056306,0.033938,0,data\avec_labels\normal\defdbef3-bea2-4f32-9d9...
1504,0.078325,0.121743,-0.039186,0.022466,0.191325,0.255515,0.132630,0.034595,0.035142,0.061571,...,0.197274,0.288624,0.011884,-0.030041,0.290409,0.166233,0.046880,-0.072580,0,data\avec_labels\normal\e30c2ce4-50bd-44a9-8db...


In [10]:
OUTPUT_DIR = Path("embeddings")
OUTPUT_DIR.mkdir(exist_ok=True)

resnetdf.to_csv(OUTPUT_DIR / "resnet50_embeddings.csv", index=False)
efnetdf.to_csv(OUTPUT_DIR  / "efficientnet_b4_embeddings.csv", index=False)

print(f"resnetdf  → {OUTPUT_DIR / 'resnet50_embeddings.csv'}")
print(f"efnetdf   → {OUTPUT_DIR / 'efficientnet_b4_embeddings.csv'}")

resnetdf  → embeddings\resnet50_embeddings.csv
efnetdf   → embeddings\efficientnet_b4_embeddings.csv


In [11]:
efnetdf.image_path


0       data\sans_label\001b158a-7af8-451e-bf31-3a9116...
1       data\sans_label\00366e8d-5520-4d3c-a70b-91a7ee...
2       data\sans_label\00455a62-f79f-4072-9a23-4951e7...
3       data\sans_label\004ce5f5-ca6b-490f-9b2f-c322c1...
4       data\sans_label\005d9a37-8894-4eb5-8367-1015d4...
                              ...                        
1501    data\avec_labels\normal\cfda6929-14c5-41b9-b75...
1502    data\avec_labels\normal\d86f39ec-55a1-4484-bcf...
1503    data\avec_labels\normal\defdbef3-bea2-4f32-9d9...
1504    data\avec_labels\normal\e30c2ce4-50bd-44a9-8db...
1505    data\avec_labels\normal\e3257bee-1143-4866-9d5...
Name: image_path, Length: 1506, dtype: object